# Node-Level Frontend Pipeline For DeepSeek V3

实现 DeepSeek V3 decoder layer 的最小前端全流程，并在 code gen pass 之后按 node 读取 `marco_op_list`，对每个 node 单独运行硬件仿真，同时对完整 `macro_op_list` 运行一次整体硬件仿真，并把 perf trace 输出到 `trace/deepseek_v3/` 目录。


## 流程
- 加载 `deepseek-v3` model card 并初始化 `DeepseekV3ModelConfig`
- 初始化 `NandConfig`、`InferenceConfig`、`MoEParallelConfig`
- 初始化 `DeepseekV3DecoderLayer`
- 使用 `NxTracer` trace 得到计算图
- 运行 `NormalizePass` 获得简化图
- 用 `build_kv_cache_state` 构造 `KVCacheState`
- 构造 `NxGraphMeta` 并写入 `graph.meta`
- 运行 `CodeGenPass`，收集全局和 node 级别的 `marco_op_list`
- 对完整 `macro_op_list` 运行一次硬件仿真，并把完整 trace 保存到 `trace/deepseek_v3/full_simulation.json`
- 对每个 node 单独运行硬件仿真，并把 node trace 保存到 `trace/deepseek_v3/` 目录
- 汇总完整仿真和 node 级别的 macro op、时延和 trace 信息


In [1]:
import json
import logging
import re
from collections import Counter
from dataclasses import fields, is_dataclass
from math import ceil
from pathlib import Path
from pprint import pprint

import torch
from Desim import SimSession
from torch.fx import GraphModule

from nandmachine.commands.macro import (
    All2AllOp,
    AllReduceOp,
    FlashMLAOp,
    MacroOp,
    MatMulOp,
    VectorOp,
)
from nandmachine.config.config import NandConfig
from nandmachine.config.hardware_config import get_device_or_raise
from nandmachine.config.inference_config import InferenceConfig, MoEParallelConfig
from nandmachine.config.model_config import DeepseekV3ModelConfig
from nandmachine.frontend.core.graph.base import NxGraphMeta, NxTracer
from nandmachine.frontend.core.passes.cod_gen import CodeGenPass
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.deepseek_v3 import DeepseekV3DecoderLayer
from nandmachine.frontend.utlis import build_kv_cache_state
from nandmachine.simulator.entry_point import run_sim
from nandmachine.simulator.hardware.xpu import xPU

MODEL_CARD_PATH = Path("model_cards/deepseek-v3.json")
DEVICE_NAME = "H100_SXM"
COMPILE_MODE = "heuristic-GPU"
LAYER_IDX = 3
NUM_RANKS = 8
ATTN_DP_SIZE = 8
ATTN_TP_SIZE = 1
FFN_TP_SIZE = 1
FFN_EP_SIZE = 8
HBM_BANDWIDTH_GBPS = 3350
TRACE_DIR = Path("trace/deepseek_v3")
FULL_TRACE_FILE_NAME = "full_simulation.json"

assert MODEL_CARD_PATH.exists(), MODEL_CARD_PATH
assert LAYER_IDX >= 0, LAYER_IDX
assert NUM_RANKS > 0, NUM_RANKS


def node_summary(graph_module: GraphModule) -> list[tuple[str, str, str]]:
    return [(node.name, node.op, str(node.target)) for node in graph_module.graph.nodes]


def macro_summary(macro_op_list: list[object]) -> Counter:
    return Counter(type(op).__name__ for op in macro_op_list)


def node_info(node: torch.fx.Node) -> dict[str, object]:
    return {
        "node_name": node.name,
        "node_op": node.op,
        "node_target": str(node.target),
    }


def _serialize_value(value: object) -> object:
    if isinstance(value, MacroOp):
        return {
            "id": value.id,
            "type": type(value).__name__,
        }
    if isinstance(value, list):
        return [_serialize_value(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_serialize_value(item) for item in value)
    return value


def macro_info(op: MacroOp) -> dict[str, object]:
    assert is_dataclass(op), type(op)
    payload = {field.name: _serialize_value(getattr(op, field.name)) for field in fields(op)}
    payload["type"] = type(op).__name__
    return payload


def _sanitize_trace_name(raw_name: str) -> str:
    return re.sub(r"[^0-9A-Za-z._-]+", "_", raw_name).strip("._")


def trace_file_path(node_index: int, node: torch.fx.Node) -> Path:
    file_name = _sanitize_trace_name(f"node_{node_index:03d}_{node.name}_{node.op}")
    return TRACE_DIR / f"{file_name}.json"


def full_trace_file_path() -> Path:
    return TRACE_DIR / FULL_TRACE_FILE_NAME


def run_macro_ops_with_trace(
    nand_config: NandConfig,
    commands: list[MacroOp],
    *,
    trace_path: Path,
    device_name: str,
    compile_mode: str,
    hbm_bandwidth_GBps: float,
) -> dict[str, object]:
    SimSession.reset()
    SimSession.init()

    hbm_bandwidth_bytes_per_sec = hbm_bandwidth_GBps * 10**9

    sim_xpu = xPU(
        nand_config,
        hbm_bandwidth_bytes_per_sec=hbm_bandwidth_bytes_per_sec,
        device_name=device_name,
        compile_mode=compile_mode,
        enable_trace=True,
    )
    sim_xpu.load_command(commands)
    SimSession.scheduler.run()

    final_time_ns = int(SimSession.sim_time.cycle)
    device = get_device_or_raise(device_name)
    final_cycle = ceil(final_time_ns * device.compute_module.clock_freq / 1e9)

    trace_path.parent.mkdir(parents=True, exist_ok=True)
    saved_trace_path = Path(sim_xpu.save_trace_file(str(trace_path)))

    tracer = sim_xpu.tracer
    assert tracer is not None
    complete_events = [event for event in tracer._events if event["ph"] == "X"]

    return {
        "cycle": final_cycle,
        "time_ns": final_time_ns,
        "trace_path": str(saved_trace_path),
        "trace_event_count": len(tracer._events),
        "trace_complete_event_count": len(complete_events),
        "trace_complete_event_categories": dict(
            Counter(event["cat"] for event in complete_events)
        ),
    }


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

model_config = DeepseekV3ModelConfig.from_dict(json.loads(MODEL_CARD_PATH.read_text()))
assert LAYER_IDX < model_config.num_hidden_layers, (LAYER_IDX, model_config.num_hidden_layers)

nand_config = NandConfig(
    num_channels=6 * 8,
    num_plane=96,
    num_block=16,
    num_pages=16,
    tRead=4000,
    tWrite=4000 * 10,
    tErase=4000 * 100,
    page_size=4,
    sram_threshold=1024 * 80,
)
parallel_config = MoEParallelConfig(
    num_ranks=NUM_RANKS,
    attn_dp_size=ATTN_DP_SIZE,
    attn_tp_size=ATTN_TP_SIZE,
    ffn_tp_size=FFN_TP_SIZE,
    ffn_ep_size=FFN_EP_SIZE,
)
inference_config = InferenceConfig(
    batch_size=1024,
    input_sequence_length=8096,
    output_sequence_length=1024,
    weight_bits=16,
    activation_bits=16,
    kv_cache_bits=16,
    kv_block_size_bytes=1024 * 256,
    memory_backend="nand",
    parallel_config=parallel_config,
)
kv_cache_state = build_kv_cache_state(nand_config, model_config, inference_config)
graph_meta = NxGraphMeta(
    nand_config=nand_config,
    model_config=model_config,
    inference_config=inference_config,
    kv_cache_state=kv_cache_state,
)
simulator_config = {
    "device_name": DEVICE_NAME,
    "compile_mode": COMPILE_MODE,
    "hbm_bandwidth_GBps": HBM_BANDWIDTH_GBPS,
}
hbm_bandwidth_bytes_per_sec = HBM_BANDWIDTH_GBPS * 10**9
runtime_simulator_config = {
    "device_name": simulator_config["device_name"],
    "compile_mode": simulator_config["compile_mode"],
    "hbm_bandwidth_GBps": simulator_config["hbm_bandwidth_GBps"],
}
TRACE_DIR.mkdir(parents=True, exist_ok=True)

print("layer index:", LAYER_IDX)
print("HBM bandwidth (GBps):", HBM_BANDWIDTH_GBPS)
print("trace output dir:", TRACE_DIR)
print("full simulation trace path:", full_trace_file_path())
print("parallel config:", parallel_config)


layer index: 3
HBM bandwidth (GBps): 3350
trace output dir: trace/deepseek_v3
full simulation trace path: trace/deepseek_v3/full_simulation.json
parallel config: MoEParallelConfig(num_ranks=8, attn_dp_size=8, attn_tp_size=1, ffn_tp_size=1, ffn_ep_size=8)


In [3]:
with torch.device("meta"):
    model = DeepseekV3DecoderLayer(
        layer_idx=LAYER_IDX,
        config=model_config,
        parallel_config=parallel_config,
    )
    graph = NxTracer().trace(model)
    graph_module = GraphModule(model, graph)

NormalizePass().transform(graph_module)
graph_module.graph.meta = {CodeGenPass.GRAPH_META_KEY: graph_meta}
CodeGenPass().transform(graph_module)

all_graph_nodes = list(graph_module.graph.nodes)
all_node_infos = [node_info(node) for node in all_graph_nodes]
macro_op_list = graph_module.graph.meta[CodeGenPass.MACRO_OP_LIST_META_KEY]
vector_ops = [op for op in macro_op_list if isinstance(op, VectorOp)]
node_macro_op_entries = []

for node in all_graph_nodes:
    if "marco_op_list" not in node.meta:
        continue
    node_macro_op_entries.append((node, node.meta["marco_op_list"]))

print("normalized node count:", len(node_summary(graph_module)))
print("node count with macro op entries:", len(node_macro_op_entries))
print("global macro op count:", len(macro_op_list))
print("global macro op type summary:", dict(macro_summary(macro_op_list)))
print("vector op type summary:", dict(Counter(op.vector_op_type for op in vector_ops)))
print("all node infos:")
for current_node_info in all_node_infos:
    pprint(current_node_info)



2026-04-06 23:29:49,479 - INFO - Processed node=input_layernorm generated_macro_ops=1
2026-04-06 23:29:49,479 - INFO - Processed node=self_attn_q_a_proj generated_macro_ops=3
2026-04-06 23:29:49,480 - INFO - Processed node=self_attn_q_a_layernorm generated_macro_ops=1
2026-04-06 23:29:49,480 - INFO - Processed node=self_attn_q_b_proj generated_macro_ops=3
2026-04-06 23:29:49,480 - INFO - Processed node=self_attn_kv_a_proj generated_macro_ops=3
2026-04-06 23:29:49,480 - INFO - Processed node=self_attn_kv_a_layernorm generated_macro_ops=1
2026-04-06 23:29:49,481 - INFO - Processed node=self_attn_attn generated_macro_ops=218
2026-04-06 23:29:49,482 - INFO - Processed node=self_attn_o_proj generated_macro_ops=9
2026-04-06 23:29:49,482 - INFO - Processed node=post_attention_layernorm generated_macro_ops=1
2026-04-06 23:29:49,483 - INFO - Processed node=mlp generated_macro_ops=231


normalized node count: 13
node count with macro op entries: 10
global macro op count: 471
global macro op type summary: {'VectorOp': 38, 'SramPrefetch': 143, 'MatMulOp': 73, 'SramPrefetchRelease': 143, 'FlashMLAOp': 72, 'All2AllOp': 2}
vector op type summary: {'rms_norm': 4, 'moe_topk_router': 1, 'silu_mul': 32, 'moe_weighted_sum': 1}
all node infos:
{'node_name': 'hidden_states',
 'node_op': 'placeholder',
 'node_target': 'hidden_states'}
{'node_name': 'input_layernorm',
 'node_op': 'call_module',
 'node_target': 'input_layernorm'}
{'node_name': 'self_attn_q_a_proj',
 'node_op': 'call_module',
 'node_target': 'self_attn.q_a_proj'}
{'node_name': 'self_attn_q_a_layernorm',
 'node_op': 'call_module',
 'node_target': 'self_attn.q_a_layernorm'}
{'node_name': 'self_attn_q_b_proj',
 'node_op': 'call_module',
 'node_target': 'self_attn.q_b_proj'}
{'node_name': 'self_attn_kv_a_proj',
 'node_op': 'call_module',
 'node_target': 'self_attn.kv_a_proj'}
{'node_name': 'self_attn_kv_a_layernorm',
 'n

In [4]:
# assert macro_op_list
# assert node_macro_op_entries
# assert any(isinstance(op, MatMulOp) for op in macro_op_list)
# assert any(isinstance(op, FlashMLAOp) for op in macro_op_list)
# # assert any(isinstance(op, AllReduceOp) for op in macro_op_list)
# assert any(isinstance(op, All2AllOp) for op in macro_op_list)
# assert any(op.vector_op_type == "rms_norm" for op in vector_ops)
# assert any(op.vector_op_type == "moe_topk_router" for op in vector_ops)
# assert any(op.vector_op_type == "moe_weighted_sum" for op in vector_ops)
# assert any(op.vector_op_type == "silu_mul" for op in vector_ops)
# assert all(all(dim > 0 for dim in op.vector_shape) for op in vector_ops)

print("deepseek v3 node-level frontend pipeline completed successfully")


deepseek v3 node-level frontend pipeline completed successfully


In [5]:
print("=" * 120)
print("full simulation")
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))

full_trace_path = full_trace_file_path()
full_sim_result = run_macro_ops_with_trace(
    nand_config,
    macro_op_list,
    trace_path=full_trace_path,
    **runtime_simulator_config,
)
assert full_sim_result["cycle"] > 0
assert full_sim_result["time_ns"] > 0
print("sim cycle:", full_sim_result["cycle"])
print("sim time ns:", full_sim_result["time_ns"])
print("trace path:", full_sim_result["trace_path"])
print("trace event count:", full_sim_result["trace_event_count"])
print("trace complete event count:", full_sim_result["trace_complete_event_count"])
print(
    "trace complete event categories:",
    full_sim_result["trace_complete_event_categories"],
)

full_run_sim_result = run_sim(
    nand_config,
    model_config,
    inference_config,
    macro_op_list,
    hbm_bandwidth_bytes_per_sec=hbm_bandwidth_bytes_per_sec,
    device_name=runtime_simulator_config["device_name"],
    compile_mode=runtime_simulator_config["compile_mode"],
)
print("run_sim result:")
pprint(
    {
        "layer_latency_ns": full_run_sim_result.layer_latency_ns,
        "model_latency_ns": full_run_sim_result.model_latency_ns,
        "model_throughput": full_run_sim_result.model_throughput,
        "kv_cache_total_size_GB": full_run_sim_result.kv_cache_total_size_GB,
    }
)

node_sim_results = []

for node_index, node in enumerate(all_graph_nodes):
    node_macro_ops = node.meta.get("marco_op_list", [])
    current_node_info = node_info(node)
    current_macro_summary = dict(macro_summary(node_macro_ops))

    print("=" * 120)
    print("node index:", node_index)
    print("node info:")
    pprint(current_node_info)
    print("macro op count:", len(node_macro_ops))
    print("macro op type summary:", current_macro_summary)

    serialized_macro_ops = [macro_info(op) for op in node_macro_ops]
    print("macro op details:")
    for macro_op_payload in serialized_macro_ops:
        pprint(macro_op_payload)

    if not node_macro_ops:
        print("simulation skipped because this node produced no macro ops")
        node_sim_results.append(
            {
                **current_node_info,
                "macro_op_count": 0,
                "macro_op_types": [],
                "sim_cycle": None,
                "sim_time_ns": None,
                "trace_path": None,
                "trace_event_count": None,
                "trace_complete_event_count": None,
                "trace_complete_event_categories": None,
                "macro_ops": serialized_macro_ops,
            }
        )
        continue

    current_trace_path = trace_file_path(node_index, node)
    sim_result = run_macro_ops_with_trace(
        nand_config,
        node_macro_ops,
        trace_path=current_trace_path,
        **runtime_simulator_config,
    )
    assert sim_result["cycle"] > 0
    assert sim_result["time_ns"] > 0
    print("sim cycle:", sim_result["cycle"])
    print("sim time ns:", sim_result["time_ns"])
    print("trace path:", sim_result["trace_path"])
    print("trace event count:", sim_result["trace_event_count"])
    print("trace complete event count:", sim_result["trace_complete_event_count"])
    print(
        "trace complete event categories:",
        sim_result["trace_complete_event_categories"],
    )

    node_sim_results.append(
        {
            **current_node_info,
            "macro_op_count": len(node_macro_ops),
            "macro_op_types": [type(op).__name__ for op in node_macro_ops],
            "sim_cycle": sim_result["cycle"],
            "sim_time_ns": sim_result["time_ns"],
            "trace_path": sim_result["trace_path"],
            "trace_event_count": sim_result["trace_event_count"],
            "trace_complete_event_count": sim_result["trace_complete_event_count"],
            "trace_complete_event_categories": sim_result[
                "trace_complete_event_categories"
            ],
            "macro_ops": serialized_macro_ops,
        }
    )


full simulation
macro op count: 471
macro op type summary: {'VectorOp': 38, 'SramPrefetch': 143, 'MatMulOp': 73, 'SramPrefetchRelease': 143, 'FlashMLAOp': 72, 'All2AllOp': 2}
VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[1024, 7168], weight_bits=16)
MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=5376), VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[1024, 7168], weight_bits=16)], dim=(1024, 7168, 1536), weight_bits=16)
VectorOp(id=5, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=5376), VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[1024, 7168], weight_bits=16)], dim=(1024, 7168, 1536), weight_bits=16)], vector_op_type='rms_norm', vector_shape=[1024, 1536], weight_bits=16)
MatMulOp(id=7, input_ops=[SramPrefetch(id=6, input_ops=[], num_prefetch_pages=18432), VectorOp(id=5, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_p

In [6]:
full_sim_result


{'cycle': 4153551,
 'time_ns': 2269700,
 'trace_path': 'trace/deepseek_v3/full_simulation.json',
 'trace_event_count': 331,
 'trace_complete_event_count': 327,
 'trace_complete_event_categories': {'compute': 183,
  'prefetch': 142,
  'transfer': 2}}

In [7]:
node_sim_results


[{'node_name': 'hidden_states',
  'node_op': 'placeholder',
  'node_target': 'hidden_states',
  'macro_op_count': 0,
  'macro_op_types': [],
  'sim_cycle': None,
  'sim_time_ns': None,
  'trace_path': None,
  'trace_event_count': None,
  'trace_complete_event_count': None,
  'trace_complete_event_categories': None,
  'macro_ops': []},
 {'node_name': 'input_layernorm',
  'node_op': 'call_module',
  'node_target': 'input_layernorm',
  'macro_op_count': 1,
  'macro_op_types': ['VectorOp'],
  'sim_cycle': 872,
  'sim_time_ns': 476,
  'trace_path': 'trace/deepseek_v3/node_001_input_layernorm_call_module.json',
  'trace_event_count': 5,
  'trace_complete_event_count': 1,
  'trace_complete_event_categories': {'compute': 1},
  'macro_ops': [{'id': 1,
    'input_ops': [],
    'vector_op_type': 'rms_norm',
    'vector_shape': [1024, 7168],
    'weight_bits': 16,
    'type': 'VectorOp'}]},
 {'node_name': 'self_attn_q_a_proj',
  'node_op': 'call_module',
  'node_target': 'self_attn.q_a_proj',
  'm

In [8]:
non_empty_node_results = [result for result in node_sim_results if result["macro_op_count"] > 0]
empty_node_results = [result for result in node_sim_results if result["macro_op_count"] == 0]

assert len(node_sim_results) == len(all_graph_nodes)
assert full_sim_result["cycle"] > 0
assert full_sim_result["time_ns"] > 0
assert full_sim_result["trace_path"] is not None
assert full_sim_result["trace_event_count"] is not None
assert full_sim_result["trace_complete_event_count"] is not None
assert non_empty_node_results
assert all(result["sim_cycle"] is not None for result in non_empty_node_results)
assert all(result["sim_time_ns"] is not None for result in non_empty_node_results)
assert all(result["trace_path"] is not None for result in non_empty_node_results)
assert all(result["trace_event_count"] is not None for result in non_empty_node_results)
assert all(result["trace_complete_event_count"] is not None for result in non_empty_node_results)
assert all(result["sim_cycle"] is None for result in empty_node_results)
assert all(result["trace_path"] is None for result in empty_node_results)

full_sim_summary = {
    "macro_op_count": len(macro_op_list),
    "macro_op_type_summary": dict(macro_summary(macro_op_list)),
    "sim_cycle": full_sim_result["cycle"],
    "sim_time_ns": full_sim_result["time_ns"],
    "trace_path": full_sim_result["trace_path"],
    "trace_event_count": full_sim_result["trace_event_count"],
    "trace_complete_event_count": full_sim_result["trace_complete_event_count"],
    "trace_complete_event_categories": full_sim_result[
        "trace_complete_event_categories"
    ],
}
run_sim_summary = {
    "layer_latency_ns": full_run_sim_result.layer_latency_ns,
    "model_latency_ns": full_run_sim_result.model_latency_ns,
    "model_throughput": full_run_sim_result.model_throughput,
    "kv_cache_total_size_GB": full_run_sim_result.kv_cache_total_size_GB,
}

summary_rows = [
    {
        "node_name": result["node_name"],
        "node_op": result["node_op"],
        "node_target": result["node_target"],
        "macro_op_count": result["macro_op_count"],
        "macro_op_types": result["macro_op_types"],
        "sim_cycle": result["sim_cycle"],
        "sim_time_ns": result["sim_time_ns"],
        "trace_path": result["trace_path"],
        "trace_event_count": result["trace_event_count"],
        "trace_complete_event_count": result["trace_complete_event_count"],
        "trace_complete_event_categories": result["trace_complete_event_categories"],
    }
    for result in node_sim_results
]

{
    "full_simulation": full_sim_summary,
    "run_simulation": run_sim_summary,
    "node_simulations": summary_rows,
}


{'full_simulation': {'macro_op_count': 471,
  'macro_op_type_summary': {'VectorOp': 38,
   'SramPrefetch': 143,
   'MatMulOp': 73,
   'SramPrefetchRelease': 143,
   'FlashMLAOp': 72,
   'All2AllOp': 2},
  'sim_cycle': 4153551,
  'sim_time_ns': 2269700,
  'trace_path': 'trace/deepseek_v3/full_simulation.json',
  'trace_event_count': 331,
  'trace_complete_event_count': 327,
  'trace_complete_event_categories': {'compute': 183,
   'prefetch': 142,
   'transfer': 2}},
 'run_simulation': {'layer_latency_ns': 2269700,
  'model_latency_ns': 138451700,
  'model_throughput': 7396.081088206212,
  'kv_cache_total_size_GB': 611.19140625},
 'node_simulations': [{'node_name': 'hidden_states',
   'node_op': 'placeholder',
   'node_target': 'hidden_states',
   'macro_op_count': 0,
   'macro_op_types': [],
   'sim_cycle': None,
   'sim_time_ns': None,
   'trace_path': None,
   'trace_event_count': None,
   'trace_complete_event_count': None,
   'trace_complete_event_categories': None},
  {'node_name':

In [9]:
# print(kv_cache_state)
